In [6]:
import numpy as np
from scipy.optimize import minimize

from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [7]:
X, y = make_classification(
    n_samples=100,
    n_features=3,
    n_informative=3,
    n_redundant=0,
    n_classes=2,
    n_clusters_per_class=1,
    class_sep=2.5,
    random_state=42
)

In [8]:
y = np.where(y == 0, -1, 1)

In [9]:
class HardMarginSVM:
    def __init__(self):
        self.w = None
        self.b = None
        self.support_vectors = None
        self.support_indices = None
        
    def fit(self , X ,y):
        n_samples , n_features = X.shape
        
        initial_theta = np.zeros(n_features + 1)
        
        def objective(theta):
            w = theta[:-1]
            return 0.5 * np.dot(w ,w)
        
        def constraints(theta):
            w = theta[:-1]
            b = theta[-1]
            
            decision = X@w+b
            return y*decision -1
        
        constraint = {
            "type" : "ineq",
            "fun" : constraints
        }
        
        result = minimize(objective , initial_theta , method = 'SLSQP' , constraints = constraint , options = {"ftol" : 1e-10, "maxiter" : 2000})
        
        if not result.success:
            raise RuntimeError(result.message)
        
        self.w = result.x[:-1]
        self.b = result.x[-1]
        
        margins = y*(X@self.w + self.b)
        self.support_indices_ = np.where(
            np.isclose(margins, 1, atol=1e-4)
        )[0]

        self.support_vectors_ = X[self.support_indices_]

        return self
    
    
    
    def decision_function(self, X):

        return X @ self.w + self.b


    def predict(self, X):

        scores = self.decision_function(X)

        return np.where(scores >= 0, 1, -1)
            

In [10]:
scratch_svm = HardMarginSVM()

scratch_svm.fit(X, y)

print("Scratch SVM")
print("w =", scratch_svm.w)
print("b =", scratch_svm.b)

print("Number of support vectors:",len(scratch_svm.support_vectors_))

Scratch SVM
w = [0.99887655 0.53958773 0.14182999]
b = 0.5809946552693998
Number of support vectors: 3


In [12]:
sklearn_svm = SVC(
    kernel="linear",
    C=1e6
)

sklearn_svm.fit(X, y)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1000000.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [13]:
print("\nSklearn SVM")

print("w =", sklearn_svm.coef_[0])
print("b =", sklearn_svm.intercept_[0])

print("Number of support vectors:",
      len(sklearn_svm.support_vectors_))


Sklearn SVM
w = [0.99868285 0.53948324 0.1418023 ]
b = 0.5809473842656466
Number of support vectors: 3


In [14]:
scratch_pred = scratch_svm.predict(X)

sklearn_pred = sklearn_svm.predict(X)

print(
    "\nScratch accuracy:",
    accuracy_score(y, scratch_pred)
)

print(
    "Sklearn accuracy:",
    accuracy_score(y, sklearn_pred)
)

print(
    "Predictions identical:",
    np.array_equal(scratch_pred, sklearn_pred)
)


Scratch accuracy: 1.0
Sklearn accuracy: 1.0
Predictions identical: True


In [ ]:
import plotly.graph_objects as go
import numpy as np


w = scratch_svm.w
b = scratch_svm.b



x_range = np.linspace(X[:, 0].min(), X[:, 0].max(), 30)
y_range = np.linspace(X[:, 1].min(), X[:, 1].max(), 30)

xx, yy = np.meshgrid(x_range, y_range)


def plane_z(offset):

    return (
        offset
        - b
        - w[0] * xx
        - w[1] * yy
    ) / w[2]


z_decision = plane_z(0)
z_positive = plane_z(1)
z_negative = plane_z(-1)



fig = go.Figure()




fig.add_trace(
    go.Surface(
        x=xx,
        y=yy,
        z=z_decision,
        opacity=0.35,
        showscale=False,
        name="Decision Plane"
    )
)





fig.add_trace(
    go.Surface(
        x=xx,
        y=yy,
        z=z_positive,
        opacity=0.25,
        showscale=False,
        name="Positive Margin"
    )
)




fig.add_trace(
    go.Surface(
        x=xx,
        y=yy,
        z=z_negative,
        opacity=0.25,
        showscale=False,
        name="Negative Margin"
    )
)



mask_neg = y == -1

fig.add_trace(
    go.Scatter3d(
        x=X[mask_neg, 0],
        y=X[mask_neg, 1],
        z=X[mask_neg, 2],
        mode="markers",
        name="Class -1",
        marker=dict(
            size=5
        )
    )
)


mask_pos = y == 1

fig.add_trace(
    go.Scatter3d(
        x=X[mask_pos, 0],
        y=X[mask_pos, 1],
        z=X[mask_pos, 2],
        mode="markers",
        name="Class +1",
        marker=dict(
            size=5
        )
    )
)


sv = scratch_svm.support_indices_

fig.add_trace(
    go.Scatter3d(
        x=X[sv, 0],
        y=X[sv, 1],
        z=X[sv, 2],
        mode="markers",
        name="Support Vectors",
        marker=dict(
            size=10,
            symbol="diamond",
            line=dict(width=2)
        )
    )
)


fig.update_layout(
    title="Hard-Margin SVM — Decision & Margin Planes",
    scene=dict(
        xaxis_title="X₁",
        yaxis_title="X₂",
        zaxis_title="X₃"
    ),
    width=900,
    height=700
)

fig.show()